In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor

1. Exploring the Dataset

In [ ]:
df = pd.read_csv("Medical_insurance.csv")
print(df.head())

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

Data Cleaning

In [ ]:
# Removing Missing Values
df = df.dropna()

# Remove duplicates if any
df = df.drop_duplicates()

Univariate Analysis

In [ ]:
# Numerical distributions
df.hist(figsize=(10,8))
plt.tight_layout()
plt.show()

# Categorical counts
sns.countplot(x='sex', data=df)
plt.show()

sns.countplot(x='smoker', data=df)
plt.show()

sns.countplot(x='region', data=df)
plt.show()

Bivariate Analysis

In [ ]:
# Charges vs Age
sns.scatterplot(x='age', y='charges', data=df)
plt.show()

# Charges vs BMI
sns.scatterplot(x='bmi', y='charges', data=df)
plt.show()

# Smoker vs Charges
sns.boxplot(x='smoker', y='charges', data=df)
plt.show()

# Correlation heatmap
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

Feature Engineering

In [ ]:
# Create BMI Category
df['bmi_category'] = pd.cut(df['bmi'],
                           bins=[0, 18.5, 25, 30, 100],
                           labels=['underweight', 'normal', 'overweight', 'obese'])

# Create Age Groups
df['age_group'] = pd.cut(df['age'],
                        bins=[0, 18, 35, 50, 65],
                        labels=['teen', 'young_adult', 'adult', 'senior'])

# Interaction Feature
df['smoker_bmi'] = df['bmi'] * (df['smoker'] == 'yes').astype(int)

# One-Hot Encoding
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region', 'bmi_category', 'age_group'], drop_first=True)

Train-Test Split

In [ ]:
X=df.drop('charges', axis=1)
y=df['charges']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Feature Scaling

In [ ]:
from sklearn.pipeline import Pipeline

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

Model Training

BaseLine Model

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)

y_base = baseline.predict(X_test)

print("Baseline Model")
print("R2:", r2_score(y_test, y_base))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_base)))
print("MAE:", mean_absolute_error(y_test, y_base))

Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Linear Regression")
print("R2:", r2_score(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))

Ridge and Lasso

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)

print("Ridge R2:", r2_score(y_test, ridge.predict(X_test)))
print("Ridge RMSE:", np.sqrt(mean_squared_error(y_test, ridge.predict(X_test))))
print("Ridge MAE:", mean_absolute_error(y_test, ridge.predict(X_test)))
print("Lasso R2:", r2_score(y_test, lasso.predict(X_test)))
print("Lasso RMSE:", np.sqrt(mean_squared_error(y_test, lasso.predict(X_test))))
print("Lasso MAE:", mean_absolute_error(y_test, lasso.predict(X_test)))

Decision Tree

In [ ]:
dt = DecisionTreeRegressor(max_depth=5)
dt.fit(X_train, y_train)

print("Decision Tree R2:", r2_score(y_test, dt.predict(X_test)))
print("Decision Tree RMSE:", np.sqrt(mean_squared_error(y_test, dt.predict(X_test))))
print("Decision Tree MAE:", mean_absolute_error(y_test, dt.predict(X_test)))

Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200,max_depth=20,min_samples_split=2, min_samples_leaf=1,random_state=42)
rf.fit(X_train, y_train)

print("Random Forest R2:", r2_score(y_test, rf.predict(X_test)))
print("Random Forest RMSE:", np.sqrt(mean_squared_error(y_test, rf.predict(X_test))))
print("Random Forest MAE:", mean_absolute_error(y_test, rf.predict(X_test)))

Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05)
gb.fit(X_train, y_train)

print("Gradient Boosting R2:", r2_score(y_test, gb.predict(X_test)))
print("Gradient Boosting RMSE:", np.sqrt(mean_squared_error(y_test, gb.predict(X_test))))
print("Gradient Boosting MAE:", mean_absolute_error(y_test, gb.predict(X_test)))

AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
ada = AdaBoostRegressor(n_estimators=100, learning_rate=0.1)
ada.fit(X_train, y_train)

print("AdaBoost R2:", r2_score(y_test, ada.predict(X_test)))
print("AdaBoost RMSE:", np.sqrt(mean_squared_error(y_test, ada.predict(X_test))))
print("AdaBoost MAE:", mean_absolute_error(y_test, ada.predict(X_test)))

XG Boost

In [ ]:
xg= XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xg.fit(X_train, y_train)

print("XGBoost R2:", r2_score(y_test, xg.predict(X_test)))
print("XGBoost RMSE:", np.sqrt(mean_squared_error(y_test, xg.predict(X_test))))
print("XGBoost MAE:", mean_absolute_error(y_test, xg.predict(X_test)))

Hyperparameter Tuning

Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_random = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    n_iter=20,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

rf_random.fit(X_train, y_train)
best_rf = rf_random.best_estimator_

print("Best RF Params:", rf_random.best_params_)
print("RF R2:", r2_score(y_test, best_rf.predict(X_test)))
print("RF RMSE:", np.sqrt(mean_squared_error(y_test, best_rf.predict(X_test))))
print("RF MAE:", mean_absolute_error(y_test, best_rf.predict(X_test)))

XG Boost

In [ ]:
from xgboost import XGBRegressor

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_random = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    xgb_params,
    n_iter=20,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

xgb_random.fit(X_train, y_train)
best_xgb = xgb_random.best_estimator_

print("Best XGB Params:", xgb_random.best_params_)
print("XGB R2:", r2_score(y_test, best_xgb.predict(X_test)))
print("XGB RMSE:", np.sqrt(mean_squared_error(y_test, best_xgb.predict(X_test))))
print("XGB MAE:", mean_absolute_error(y_test, best_xgb.predict(X_test)))

Gradient Boosting

In [ ]:
gb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0]
}

gb_random = RandomizedSearchCV(
    GradientBoostingRegressor(),
    gb_params,
    n_iter=20,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

gb_random.fit(X_train, y_train)
best_gb = gb_random.best_estimator_

print("Best GB Params:", gb_random.best_params_)
print("GB R2:", r2_score(y_test, best_gb.predict(X_test)))
print("GB RMSE:", np.sqrt(mean_squared_error(y_test, best_gb.predict(X_test))))
print("GB MAE:", mean_absolute_error(y_test, best_gb.predict(X_test)))

Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_gb, X_train, y_train, cv=5, scoring='r2')

print("Gradient Boosting")
print("Cross Validation R2 Scores:", cv_scores)
print("Mean CV R2:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='r2')

print("Random Forest")
print("Cross Validation R2 Scores:", cv_scores)
print("Mean CV R2:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_xgb, X_train, y_train, cv=5, scoring='r2')


print("XGBoost")
print("Cross Validation R2 Scores:", cv_scores)
print("Mean CV R2:", np.mean(cv_scores))
print("Std Dev:", np.std(cv_scores))

Stacking

In [ ]:
from sklearn.ensemble import StackingRegressor

estimators = [
    ('rf', best_rf),
    ('gb', best_gb),
    ('xgb', best_xgb)
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    n_jobs=-1
)

stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict(X_test)

print("Stacking R2:", r2_score(y_test, stack_pred))
print("Stacking RMSE:", np.sqrt(mean_squared_error(y_test, stack_pred)))
print("Stacking MAE:", mean_absolute_error(y_test, stack_pred))

In [ ]:
from sklearn.linear_model import Ridge

stack_model = StackingRegressor(
    estimators=[
        ('rf', best_rf),
        ('gb', best_gb),
        ('xgb', best_xgb)
    ],
    final_estimator=Ridge(alpha=10),
    passthrough=True
)

stack_model.fit(X_train, y_train)
stack_pred = stack_model.predict(X_test)
print("Stacking with Ridge R2:", r2_score(y_test, stack_pred))
print("Stacking with Ridge RMSE:", np.sqrt(mean_squared_error(y_test, stack_pred)))
print("Stacking with Ridge MAE:", mean_absolute_error(y_test, stack_pred))

Saving the Best Model

In [ ]:
import pickle

# Save the trained stacking model
with open("stacking_model.pkl", "wb") as f:
    pickle.dump(stack_model, f)

In [ ]:
with open("model_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)